Lib

In [58]:
import requests
import pandas as pd
from tqdm import tqdm


QUERIES

In [59]:
topics = [
    "diversity",
    "inclusion",
    "equity",
    "LGBT",
    "LGBTQ",
    "gender identity",
    "transgender",
    "transsexual",
    "disability",
    "neurodiversity",
    "neurodivergent",
    "adhd",
    "autism",
    "sexual orientation",
    "disability accessibility"
]

context = [
    "workplace",
    "organization",
    "organizational",
    "employee"
]

In [60]:
queries = []

for t in topics:
    for c in context:
        queries.append(f"{t} {c}")

search_query = (" OR ").join(queries)

In [61]:
filter = "publication_year:2000-2025,type:article,primary_topic.field.id:14|18|20|33"

API REQUEST

In [62]:
url = "https://api.openalex.org/works"    

params = {
    "search": search_query, 
    "filter": filter,
    "per_page": 200,
    "cursor": "*"
}

RECORDS

In [ ]:
def clean_text(text):
    if text:
        return text.replace("\u2028", " ").replace("\u2029", " ")
    return text

In [ ]:
records = []

for _ in tqdm(range(100)):   #100 pages
    response = requests.get(url, params=params)
    data = response.json()

    if "results" not in data:
        print(data)
        break
    
    for work in data["results"]:
        title = work.get("title")
        year = work.get("publication_year")
        cited = work.get("cited_by_count")

        records.append({
            "title": clean_text(title),
            "year": year,
            "citations": cited
        })

    next_cursor = data["meta"].get("next_cursor")

    if not next_cursor:
        print("End of results")
        break

    params["cursor"] = next_cursor

 46%|████▌     | 46/100 [02:34<03:00,  3.35s/it]

End of results


DATASET

In [66]:
df = pd.DataFrame(records)
df.head()

,title,year,citations
0,"Neurodiversity, Gender, and Work",2024,5
1,Economic Perspectives on Corporate Social Resp...,2012,1093
2,The Participation of People with Disabilities ...,2019,430
3,"Diversity, Inclusion and Leadership: Perspecti...",2020,4
4,"Voice, silence, and diversity in 21st century ...",2011,292


CSV:

In [67]:
df.to_csv("../data/raw/openalex_workplace_inclusion.csv", index=False)